In [1]:
import os
import sys

import torch
from torchvision import transforms
from torchvision.datasets import UCF101
from tqdm import tqdm 
from PIL import Image

sys.path.append("..")
from datasets import video_transforms

In [2]:
base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [3]:
# Create output directory
output_dir = os.path.join(base_dir, "datasets/ucf101/UCF101/UCF-101-train")
os.makedirs(output_dir, exist_ok=True)

# Define Arguments
class Args:
    root_path = os.path.join(base_dir, "datasets/ucf101/UCF101/UCF-101")
    annotation_path = os.path.join(base_dir, "datasets/ucf101/UCF101TrainTestSplits-RecognitionTask/ucfTrainTestlist")

    image_h = 240
    image_w = 320
    frames_per_clip = 16
    step_between_clips = 8
    
args = Args()

In [4]:
ucf_dataset = UCF101(root=args.root_path, 
                     annotation_path=args.annotation_path, 
                     frames_per_clip=args.frames_per_clip, 
                     step_between_clips=args.step_between_clips, 
                     train=True, 
                     num_workers=8)
print(f"Total clips in train dataset: {len(ucf_dataset)}")

100%|██████████████████████████████████████████████████████████████████████████████████████████████| 833/833 [00:25<00:00, 32.06it/s]


Total clips in train dataset: 209582


In [5]:
def get_frames(video, args):
    transform_video = transforms.Compose([
            video_transforms.ToTensorVideo(), # TCHW
            video_transforms.ResizeVideo((args.image_h, args.image_w)),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], inplace=True)
    ])
    
    # Convert to PyTorch tensor and permute dimensions
    video = video.permute(0, 3, 1, 2)  # TCHW format
            
    video = transform_video(video) #.float())  # Apply transformations

    return video

In [7]:
# Loop through the dataset
for idx, (video, audio, label) in tqdm(enumerate(ucf_dataset), total= len(ucf_dataset)):
    class_name = f"{ucf_dataset.classes[label]}"  # You may replace this with actual class names
    video_name = f"video_{idx}"

    # Create directories
    class_dir = os.path.join(output_dir, class_name)
    video_dir = os.path.join(class_dir, video_name)
    os.makedirs(video_dir, exist_ok=True)

    # Process video frames
    processed_frames = get_frames(video, args)  # Shape (T, C, H, W)

    # Save frames as PNG images
    for frame_idx in range(processed_frames.shape[0]):
        frame = processed_frames[frame_idx].permute(1, 2, 0).cpu().numpy()  # Convert back to HWC format
        frame = (frame * 255).clip(0, 255).astype('uint8')  # Convert from [-1,1] to [0,255]
        frame_image = Image.fromarray(frame)

        # Save frame as PNG
        frame_path = os.path.join(video_dir, f"frame_{frame_idx:04d}.png")
        frame_image.save(frame_path)

print(f"Frames saved in {output_dir}")

  0%|                                                                                        | 162/209582 [00:29<10:35:13,  5.49it/s]


KeyboardInterrupt: 